# 🧠 NeuroScan-AI: U-Net Brain Tumor Segmentation
## Task: Tumor Segmentation (Pixel-wise Mask)
### Dataset: BraTS 2021 | Model: U-Net | Tracking: MLflow + DagsHub

---
**Classes:**
- 0: Background
- 1: Necrotic Core (NCR)
- 2: Peritumoral Edema (ED)
- 3: Enhancing Tumor (ET)

**Modalities:** T1ce + T2 + FLAIR  
**Image Size:** 128x128  
**Epochs:** 50 | Batch: 16 | GPU: Kaggle T4 x2

In [1]:
# ============================================================
# CELL 2: Install Dependencies
# NeuroScan-AI — U-Net Brain Tumor Segmentation
# ============================================================

!pip install dagshub mlflow nibabel opencv-python-headless scikit-learn -q
!pip install segmentation-models-pytorch albumentations -q

import gc
import torch
import psutil

# GPU Info
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
print(f"✅ GPUs: {torch.cuda.device_count()}")

# RAM Info
ram = psutil.virtual_memory()
print(f"✅ RAM Total: {ram.total / 1e9:.2f} GB")
print(f"✅ RAM Available: {ram.available / 1e9:.2f} GB")

print("✅ All dependencies installed!")

# Cleanup
gc.collect()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 82.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 41.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

41

In [2]:
# ============================================================
# CELL 3: Dagshub + MLflow Setup
# NeuroScan-AI — U-Net Brain Tumor Segmentation
# ============================================================

import os
import mlflow
from kaggle_secrets import UserSecretsClient

try:
    # ── Secrets ────────────────────────────────────────────
    secrets = UserSecretsClient()
    dagshub_token = secrets.get_secret("DAGSHUB_TOKEN")

    os.environ["MLFLOW_TRACKING_USERNAME"] = "kaushik-chariya"
    os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token
    os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow"

    mlflow.set_tracking_uri("https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow")
    mlflow.set_experiment("UNet-Brain-Tumor-Segmentation")

    # ── Tags ───────────────────────────────────────────────
    mlflow.set_tags({
        "model"       : "U-Net",
        "dataset"     : "BraTS2021",
        "modalities"  : "T1+T1ce+T2+FLAIR",
        "classes"     : "background+necrotic+edema+enhancing",
        "engineer"    : "kaushik-chariya",
        "project"     : "NeuroScan-AI",
        "environment" : "Kaggle-T4x2",
        "task"        : "Tumor-Segmentation",
    })

    print("✅ MLflow + Dagshub Connected!")
    print("✅ Tags Set!")
    print("🔗 https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow")

except Exception as e:
    print(f"❌ Connection Failed: {e}")
    raise

✅ MLflow + Dagshub Connected!
✅ Tags Set!
🔗 https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow


In [3]:
# ============================================================
# CELL 4: Extract BraTS 2021 Dataset
# NeuroScan-AI — U-Net Brain Tumor Segmentation
# ============================================================

import tarfile
import os
import gc

TAR_PATH     = '/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar'
EXTRACT_PATH = '/kaggle/working/BraTS2021'

os.makedirs(EXTRACT_PATH, exist_ok=True)

try:
    existing = os.listdir(EXTRACT_PATH)

    if len(existing) > 0:
        print(f"✅ Already extracted! {len(existing)} patients found")
    else:
        print("⏳ Extracting BraTS 2021... (5-10 min)")
        with tarfile.open(TAR_PATH, 'r') as tar:
            tar.extractall(EXTRACT_PATH)
        print("✅ Extraction Done!")

    # ── Patient Count ──────────────────────────────────────
    patients = sorted([
        p for p in os.listdir(EXTRACT_PATH)
        if os.path.isdir(os.path.join(EXTRACT_PATH, p))
    ])

    print(f"📁 Total Patients : {len(patients)}")
    print(f"📂 Sample Patient : {patients[0]}")
    print(f"📂 Last  Patient  : {patients[-1]}")

except Exception as e:
    print(f"❌ Extraction Failed: {e}")
    raise

finally:
    gc.collect()

⏳ Extracting BraTS 2021... (5-10 min)


/tmp/ipykernel_58/147819976.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(EXTRACT_PATH)


✅ Extraction Done!
📁 Total Patients : 1251
📂 Sample Patient : BraTS2021_00000
📂 Last  Patient  : BraTS2021_01666


In [4]:
# ============================================================
# CELL 5: Load Patients
# NeuroScan-AI — U-Net Brain Tumor Segmentation
# ============================================================

import os
import gc

DATASET_PATH = "/kaggle/working/BraTS2021"

try:
    patients = sorted([
        p for p in os.listdir(DATASET_PATH)
        if os.path.isdir(os.path.join(DATASET_PATH, p))
    ])

    assert len(patients) > 0, "❌ No patients found!"

    print(f"✅ Total Patients : {len(patients)}")
    print(f"📂 First 3        : {patients[:3]}")
    print(f"📂 Last  3        : {patients[-3:]}")

    # ── Verify Modalities ──────────────────────────────────
    sample     = patients[0]
    sample_dir = os.path.join(DATASET_PATH, sample)
    files      = os.listdir(sample_dir)

    print(f"\n📁 Sample Patient : {sample}")
    print(f"   Files Found    : {len(files)}")
    for f in sorted(files):
        print(f"   → {f}")

except Exception as e:
    print(f"❌ Error: {e}")
    raise

finally:
    gc.collect()

✅ Total Patients : 1251
📂 First 3        : ['BraTS2021_00000', 'BraTS2021_00002', 'BraTS2021_00003']
📂 Last  3        : ['BraTS2021_01664', 'BraTS2021_01665', 'BraTS2021_01666']

📁 Sample Patient : BraTS2021_00000
   Files Found    : 5
   → BraTS2021_00000_flair.nii.gz
   → BraTS2021_00000_seg.nii.gz
   → BraTS2021_00000_t1.nii.gz
   → BraTS2021_00000_t1ce.nii.gz
   → BraTS2021_00000_t2.nii.gz


In [5]:
# ============================================================
# CELL 6: UNet Dataset Preparation — Hospital Grade
# NeuroScan-AI — U-Net Brain Tumor Segmentation
# Z-score + CLAHE + Best Slices + Parallel Processing
# ============================================================

import numpy as np
import nibabel as nib
import cv2
import os
import gc
import psutil
from multiprocessing import Pool, cpu_count

# ── Config ─────────────────────────────────────────────────
IMAGE_SIZE   = 256
OUTPUT_PATH  = "/kaggle/working/unet_dataset"
DATASET_PATH = "/kaggle/working/BraTS2021"
N_SLICES     = 5

os.makedirs(f"{OUTPUT_PATH}/images", exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/masks",  exist_ok=True)

# ── Normalization ───────────────────────────────────────────
def norm(x: np.ndarray) -> np.ndarray:
    """Z-score normalization — better than min-max for MRI."""
    mean, std = x.mean(), x.std()
    x = (x - mean) / (std + 1e-8)
    x = np.clip(x, -3, 3)
    x = (x + 3) / 6
    return x

# ── CLAHE Enhancement ───────────────────────────────────────
def apply_clahe(channel: np.ndarray) -> np.ndarray:
    """Contrast Limited Adaptive Histogram Equalization."""
    ch    = (channel * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    return clahe.apply(ch)

# ── Best Tumor Slices ───────────────────────────────────────
def get_best_slices(seg_vol: np.ndarray, n: int = 5) -> list:
    """Select slices with maximum tumor area."""
    tumor_mask = (seg_vol > 0)
    counts     = tumor_mask.sum(axis=(0, 1))
    best       = np.argsort(counts)[::-1][:n]
    return sorted(best.tolist())

# ── Process Single Patient ──────────────────────────────────
def process_patient(args: tuple) -> int:
    patient, dataset_path, output_path = args
    patient_path = os.path.join(dataset_path, patient)
    saved = 0

    try:
        files = os.listdir(patient_path)

        t1ce_f  = [f for f in files if "t1ce"  in f and f.endswith(".nii.gz")]
        t2_f    = [f for f in files if "t2"    in f and f.endswith(".nii.gz")]
        flair_f = [f for f in files if "flair" in f and f.endswith(".nii.gz")]
        seg_f   = [f for f in files if "seg"   in f and f.endswith(".nii.gz")]

        # ── Skip incomplete patients ────────────────────────
        if not all([t1ce_f, t2_f, flair_f, seg_f]):
            return 0

        # ── Load Volumes ────────────────────────────────────
        t1ce_vol  = nib.load(os.path.join(patient_path, t1ce_f[0])).get_fdata()
        t2_vol    = nib.load(os.path.join(patient_path, t2_f[0])).get_fdata()
        flair_vol = nib.load(os.path.join(patient_path, flair_f[0])).get_fdata()
        seg_vol   = nib.load(os.path.join(patient_path, seg_f[0])).get_fdata()

        # ── Z-score Normalization ───────────────────────────
        t1ce_vol  = norm(t1ce_vol)
        t2_vol    = norm(t2_vol)
        flair_vol = norm(flair_vol)

        # ── Best Tumor Slices ───────────────────────────────
        slice_indices = get_best_slices(seg_vol, n=N_SLICES)

        for idx in slice_indices:
            if idx >= seg_vol.shape[2]:
                continue

            # ── CLAHE per channel ───────────────────────────
            t1ce_s  = apply_clahe(t1ce_vol[:, :, idx])
            t2_s    = apply_clahe(t2_vol[:, :, idx])
            flair_s = apply_clahe(flair_vol[:, :, idx])

            # ── Stack 3 channel image ───────────────────────
            img = np.stack([t1ce_s, t2_s, flair_s], axis=-1)
            img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))

            # ── 4 class mask ────────────────────────────────
            seg_slice = seg_vol[:, :, idx]
            mask      = np.zeros_like(seg_slice, dtype=np.uint8)
            mask[seg_slice == 1] = 1  # Necrotic Core
            mask[seg_slice == 2] = 2  # Peritumoral Edema
            mask[seg_slice == 4] = 3  # Enhancing Tumor
            mask = cv2.resize(
                mask,
                (IMAGE_SIZE, IMAGE_SIZE),
                interpolation=cv2.INTER_NEAREST
            )

            # ── Save ────────────────────────────────────────
            fname = f"{patient}_slice{idx}"
            cv2.imwrite(f"{output_path}/images/{fname}.jpg", img)
            np.save(f"{output_path}/masks/{fname}.npy", mask)
            saved += 1

        # ── RAM cleanup per patient ─────────────────────────
        del t1ce_vol, t2_vol, flair_vol, seg_vol
        gc.collect()

    except Exception as e:
        print(f"⚠️  {patient}: {e}")
        return 0

    return saved

# ── Parallel Processing ─────────────────────────────────────
print(f"⚡ CPUs Available : {cpu_count()}")
print(f"🧠 RAM Available  : {psutil.virtual_memory().available / 1e9:.2f} GB")
print(f"👥 Total Patients : {len(patients)}")
print(f"📸 Slices/Patient : {N_SLICES}")
print(f"🖼️  Image Size     : {IMAGE_SIZE}x{IMAGE_SIZE}")
print("─" * 50)

args        = [(p, DATASET_PATH, OUTPUT_PATH) for p in patients]
total_saved = 0

with Pool(processes=4) as pool:
    for i, result in enumerate(pool.imap(process_patient, args)):
        total_saved += result
        if (i + 1) % 100 == 0:
            ram = psutil.virtual_memory()
            print(
                f"✅ [{i+1:4d}/{len(patients)}] "
                f"Saved: {total_saved:5d} | "
                f"RAM: {ram.available / 1e9:.1f} GB free"
            )

# ── Final Summary ───────────────────────────────────────────
print("─" * 50)
print(f"✅ Total Slices Saved  : {total_saved}")
print(f"📊 Avg Slices/Patient  : {total_saved / len(patients):.1f}")
print(f"💾 Images Dir          : {OUTPUT_PATH}/images")
print(f"💾 Masks  Dir          : {OUTPUT_PATH}/masks")

gc.collect()

⚡ CPUs Available : 4
🧠 RAM Available  : 31.89 GB
👥 Total Patients : 1251
📸 Slices/Patient : 5
🖼️  Image Size     : 256x256
──────────────────────────────────────────────────
✅ [ 100/1251] Saved:   500 | RAM: 31.1 GB free
✅ [ 200/1251] Saved:  1000 | RAM: 31.2 GB free
✅ [ 300/1251] Saved:  1500 | RAM: 30.8 GB free
✅ [ 400/1251] Saved:  2000 | RAM: 30.4 GB free
✅ [ 500/1251] Saved:  2500 | RAM: 30.6 GB free
✅ [ 600/1251] Saved:  3000 | RAM: 30.8 GB free
✅ [ 700/1251] Saved:  3500 | RAM: 30.7 GB free
✅ [ 800/1251] Saved:  4000 | RAM: 30.7 GB free
✅ [ 900/1251] Saved:  4500 | RAM: 30.5 GB free
✅ [1000/1251] Saved:  5000 | RAM: 30.7 GB free
✅ [1100/1251] Saved:  5500 | RAM: 31.0 GB free
✅ [1200/1251] Saved:  6000 | RAM: 30.6 GB free
──────────────────────────────────────────────────
✅ Total Slices Saved  : 6255
📊 Avg Slices/Patient  : 5.0
💾 Images Dir          : /kaggle/working/unet_dataset/images
💾 Masks  Dir          : /kaggle/working/unet_dataset/masks


30

In [6]:
# ============================================================
# CELL 7: Train/Val Split — Patient Level (No Data Leakage)
# NeuroScan-AI — U-Net Brain Tumor Segmentation
# ============================================================

import os
import random
import shutil
import gc

random.seed(42)

# ── Patient-Level Split (No Leakage!) ───────────────────────
# Same patient ke slices train + val dono me nahi jayenge
unique_patients = sorted(set(
    img.rsplit("_slice", 1)[0]
    for img in os.listdir(f"{OUTPUT_PATH}/images")
))

random.shuffle(unique_patients)
split          = int(0.8 * len(unique_patients))
train_patients = set(unique_patients[:split])
val_patients   = set(unique_patients[split:])

# ── Create Dirs ─────────────────────────────────────────────
for split_name in ["train", "val"]:
    os.makedirs(f"{OUTPUT_PATH}/{split_name}/images", exist_ok=True)
    os.makedirs(f"{OUTPUT_PATH}/{split_name}/masks",  exist_ok=True)

# ── Safe Copy ───────────────────────────────────────────────
def safe_copy(src: str, dst: str) -> None:
    if os.path.exists(src):
        shutil.copy(src, dst)

# ── Split Images ────────────────────────────────────────────
train_imgs, val_imgs = [], []

all_images = sorted(os.listdir(f"{OUTPUT_PATH}/images"))

for img in all_images:
    patient_id = img.rsplit("_slice", 1)[0]
    mask       = img.replace(".jpg", ".npy")

    if patient_id in train_patients:
        safe_copy(
            f"{OUTPUT_PATH}/images/{img}",
            f"{OUTPUT_PATH}/train/images/{img}"
        )
        safe_copy(
            f"{OUTPUT_PATH}/masks/{mask}",
            f"{OUTPUT_PATH}/train/masks/{mask}"
        )
        train_imgs.append(img)

    else:
        safe_copy(
            f"{OUTPUT_PATH}/images/{img}",
            f"{OUTPUT_PATH}/val/images/{img}"
        )
        safe_copy(
            f"{OUTPUT_PATH}/masks/{mask}",
            f"{OUTPUT_PATH}/val/masks/{mask}"
        )
        val_imgs.append(img)

# ── Summary ─────────────────────────────────────────────────
print(f"✅ Total   Patients : {len(unique_patients)}")
print(f"✅ Train   Patients : {len(train_patients)}")
print(f"✅ Val     Patients : {len(val_patients)}")
print("─" * 40)
print(f"✅ Total   Slices   : {len(all_images)}")
print(f"✅ Train   Slices   : {len(train_imgs)}")
print(f"✅ Val     Slices   : {len(val_imgs)}")
print(f"📊 Split Ratio      : {len(train_imgs)/len(all_images)*100:.1f}% / {len(val_imgs)/len(all_images)*100:.1f}%")

gc.collect()

✅ Total   Patients : 1251
✅ Train   Patients : 1000
✅ Val     Patients : 251
────────────────────────────────────────
✅ Total   Slices   : 6255
✅ Train   Slices   : 5000
✅ Val     Slices   : 1255
📊 Split Ratio      : 79.9% / 20.1%


0

In [7]:
# CELL 8: PyTorch Dataset Class — RAM Optimized
# NeuroScan-AI — U-Net Brain Tumor Segmentation

import os
import torch
import numpy as np
import cv2
import gc
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

class BraTSDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        mask_name = img_name.replace(".jpg", ".npy")

        img = cv2.imread(os.path.join(self.img_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = np.load(os.path.join(self.mask_dir, mask_name))
        mask = mask.astype(np.uint8)

        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img = aug["image"]
            mask = aug["mask"]

        return img, mask.long()

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ElasticTransform(p=0.3),
    A.GridDistortion(p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_dataset = BraTSDataset(
    img_dir=f"{OUTPUT_PATH}/train/images",
    mask_dir=f"{OUTPUT_PATH}/train/masks",
    transform=train_transform
)

val_dataset = BraTSDataset(
    img_dir=f"{OUTPUT_PATH}/val/images",
    mask_dir=f"{OUTPUT_PATH}/val/masks",
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

print(f"✅ Train Dataset : {len(train_dataset)} slices")
print(f"✅ Val Dataset   : {len(val_dataset)} slices")
print(f"✅ Train Batches : {len(train_loader)}")
print(f"✅ Val Batches   : {len(val_loader)}")

imgs, masks = next(iter(train_loader))
print(f"\n📐 Image Shape  : {imgs.shape}")
print(f"📐 Mask Shape   : {masks.shape}")
print(f"🎯 Mask Classes : {masks.unique()}")

del imgs, masks
gc.collect()

✅ Train Dataset : 5991 slices
✅ Val Dataset   : 2242 slices
✅ Train Batches : 748
✅ Val Batches   : 281

📐 Image Shape  : torch.Size([8, 3, 256, 256])
📐 Mask Shape   : torch.Size([8, 256, 256])
🎯 Mask Classes : tensor([0, 1, 2, 3])


148

In [8]:
# CELL 9: UNet++ ResNet34 Model — RAM Optimized
# NeuroScan-AI — U-Net Brain Tumor Segmentation

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
import gc

# ── Device ──────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {device}")
print(f"✅ GPUs   : {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"✅ GPU 0  : {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Model ───────────────────────────────────────────────────
model = smp.UnetPlusPlus(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=4,
    activation=None
)

model = model.to(device)

# ── Loss ────────────────────────────────────────────────────
dice_loss  = smp.losses.DiceLoss(mode='multiclass', smooth=1.0)
focal_loss = smp.losses.FocalLoss(mode='multiclass', gamma=2.0)

def combined_loss(pred, target):
    return dice_loss(pred, target) + 0.5 * focal_loss(pred, target)

# ── Optimizer + Scheduler ───────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=60,
    eta_min=1e-6   # ✅ LR floor — zero pe nahi jayega
)

# ── Model Info ──────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

print(f"\n✅ Model          : UNet++ ResNet34")
print(f"✅ Total Params   : {total_params:.1f}M")
print(f"✅ Trainable      : {trainable_params:.1f}M")
print(f"✅ Loss           : Dice + 0.5 × Focal")
print(f"✅ Optimizer      : AdamW (lr=1e-3, wd=1e-4)")
print(f"✅ Scheduler      : CosineAnnealingLR (eta_min=1e-6)")

gc.collect()
torch.cuda.empty_cache()

✅ Device : cuda
✅ GPUs   : 2
✅ GPU 0  : Tesla T4
✅ VRAM   : 15.6 GB


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]


✅ Model          : UNet++ ResNet34
✅ Total Params   : 26.1M
✅ Trainable      : 26.1M
✅ Loss           : Dice + 0.5 × Focal
✅ Optimizer      : AdamW (lr=1e-3, wd=1e-4)
✅ Scheduler      : CosineAnnealingLR (eta_min=1e-6)


In [9]:
# CELL 10: Disk Cleanup — Free Space Before Training
# NeuroScan-AI — U-Net Brain Tumor Segmentation

import shutil
import os
import gc

# ── Before ──────────────────────────────────────────────────
du = shutil.disk_usage("/kaggle/working")
print(f"📦 Before — Free: {du.free / 1e9:.1f} GB")

# ── Delete BraTS Raw Data ────────────────────────────────────
RAW_PATH = "/kaggle/working/BraTS2021"

try:
    if os.path.exists(RAW_PATH):
        shutil.rmtree(RAW_PATH)
        print("✅ BraTS raw data deleted!")
    else:
        print("⚠️  BraTS raw data already deleted!")
except Exception as e:
    print(f"❌ Delete failed: {e}")
    raise

# ── After ───────────────────────────────────────────────────
du = shutil.disk_usage("/kaggle/working")
print(f"✅ After  — Free: {du.free / 1e9:.1f} GB")

gc.collect()
torch.cuda.empty_cache()

📦 Before — Free: 6.1 GB
✅ BraTS raw data deleted!
✅ After  — Free: 19.5 GB


In [ ]:
# CELL 11: Training Loop — Hospital Grade
# NeuroScan-AI — U-Net Brain Tumor Segmentation

import mlflow
mlflow.end_run()

import torch
import numpy as np
import gc
from tqdm import tqdm

# ── Metrics ─────────────────────────────────────────────────
def dice_score(pred, target, num_classes=4):
    pred = torch.argmax(pred, dim=1)
    dice = 0
    count = 0
    for cls in range(1, num_classes):
        pred_cls = (pred == cls).float()
        target_cls = (target == cls).float()
        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()
        if union == 0:
            continue
        dice += (2 * intersection + 1e-8) / (union + 1e-8)
        count += 1
    return dice / count if count > 0 else torch.tensor(0.0)

def iou_score(pred, target, num_classes=4):
    pred = torch.argmax(pred, dim=1)
    iou = 0
    count = 0
    for cls in range(1, num_classes):
        pred_cls = (pred == cls).float()
        target_cls = (target == cls).float()
        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum() - intersection
        if union == 0:
            continue
        iou += (intersection + 1e-8) / (union + 1e-8)
        count += 1
    return iou / count if count > 0 else torch.tensor(0.0)

def per_class_dice(pred, target, num_classes=4):
    pred = torch.argmax(pred, dim=1)
    scores = {}
    names = {1: "necrotic", 2: "edema", 3: "enhancing"}
    for cls in range(1, num_classes):
        pred_cls = (pred == cls).float()
        target_cls = (target == cls).float()
        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()
        if union == 0:
            scores[names[cls]] = 0.0
        else:
            scores[names[cls]] = ((2 * intersection + 1e-8) / (union + 1e-8)).item()
    return scores

# ── Config ──────────────────────────────────────────────────
EPOCHS = 100
best_dice = 0.0
MODEL_PATH = "/kaggle/working/unet_best.pt"
LAST_PATH = "/kaggle/working/unet_last.pt"
CKPT_PATH = "/kaggle/working/checkpoint_latest.pt"

# ── Training ────────────────────────────────────────────────
with mlflow.start_run(run_name="UNet-BraTS2021-ResNet34-V1"):

    mlflow.set_tags({
        "model": "UNet++ ResNet34",
        "dataset": "BraTS2021",
        "modalities": "T1ce+T2+FLAIR",
        "engineer": "kaushik-chariya",
        "project": "NeuroScan-AI",
        "environment": "Kaggle-T4x2",
        "task": "Tumor-Segmentation",
    })

    mlflow.log_params({
        "model": "UNet++ ResNet34",
        "epochs": EPOCHS,
        "batch": 8,
        "image_size": 256,
        "optimizer": "AdamW",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss": "Dice+Focal",
        "dataset": "BraTS2021",
        "modalities": "T1ce+T2+FLAIR",
        "classes": "background,necrotic,edema,enhancing",
        "augmentation": "flip+rotate+elastic+distortion",
        "train_slices": len(train_dataset),
        "val_slices": len(val_dataset),
    })

    for epoch in range(1, EPOCHS + 1):

        # ── Train ────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        train_dice = 0.0

        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} Train", leave=False):
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad()
            preds = model(imgs)
            loss = combined_loss(preds, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            train_dice += dice_score(preds, masks).item()
            del imgs, masks, preds, loss

        scheduler.step()

        # ── Validation ──────────────────────────────────────
        model.eval()
        val_loss = 0.0
        val_dice = 0.0
        val_iou = 0.0
        cls_dice = {"necrotic": 0.0, "edema": 0.0, "enhancing": 0.0}

        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} Val", leave=False):
                imgs = imgs.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)
                preds = model(imgs)

                val_loss += combined_loss(preds, masks).item()
                val_dice += dice_score(preds, masks).item()
                val_iou += iou_score(preds, masks).item()

                batch_cls = per_class_dice(preds, masks)
                for k in cls_dice:
                    cls_dice[k] += batch_cls[k]

                del imgs, masks, preds

        # ── Averages ─────────────────────────────────────────
        train_loss /= len(train_loader)
        train_dice /= len(train_loader)
        val_loss /= len(val_loader)
        val_dice /= len(val_loader)
        val_iou /= len(val_loader)
        for k in cls_dice:
            cls_dice[k] /= len(val_loader)

        lr_now = optimizer.param_groups[0]["lr"]

        # ── MLflow Log ───────────────────────────────────────
        mlflow.log_metrics({
            "train_loss": round(train_loss, 4),
            "train_dice": round(train_dice, 4),
            "val_loss": round(val_loss, 4),
            "val_dice": round(val_dice, 4),
            "val_iou": round(val_iou, 4),
            "lr": round(lr_now, 6),
            "dice_necrotic": round(cls_dice["necrotic"], 4),
            "dice_edema": round(cls_dice["edema"], 4),
            "dice_enhancing": round(cls_dice["enhancing"], 4),
        }, step=epoch)

        # ── Print ────────────────────────────────────────────
        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {train_loss:.4f} | "
            f"Train Dice: {train_dice:.4f} | "
            f"Val Dice: {val_dice:.4f} | "
            f"Val IoU: {val_iou:.4f} | "
            f"LR: {lr_now:.6f}"
        )
        print(
            f"         | "
            f"Necrotic: {cls_dice['necrotic']:.4f} | "
            f"Edema: {cls_dice['edema']:.4f} | "
            f"Enhancing: {cls_dice['enhancing']:.4f}"
        )

        # ── Best Model Save ───────────────────────────────────
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), MODEL_PATH)
            mlflow.log_metric("best_val_dice", best_dice, step=epoch)
            print(f"  💾 Best model saved! Dice: {best_dice:.4f}")

        # ── Last Model Save ───────────────────────────────────
        torch.save(model.state_dict(), LAST_PATH)

        # ✅ Har 10 epochs pe checkpoint — OVERRIDE (space safe!)
        if epoch % 10 == 0:
            torch.save(model.state_dict(), CKPT_PATH)
            print(f"  📦 Checkpoint saved! Epoch {epoch}")

        # ── RAM + VRAM Cleanup ────────────────────────────────
        gc.collect()
        torch.cuda.empty_cache()

    # ── Final ────────────────────────────────────────────────
    mlflow.log_artifact(MODEL_PATH)
    mlflow.log_artifact(LAST_PATH)

    print(f"\n{'='*55}")
    print(f"✅ Best Val Dice  : {best_dice:.4f}")
    print(f"✅ Best model     : {MODEL_PATH}")
    print(f"✅ Last model     : {LAST_PATH}")
    print(f"✅ Logged to MLflow + DagsHub!")
    print(f"{'='*55}")

🏃 View run illustrious-calf-270 at: https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow/#/experiments/3/runs/01c9ac43f2c241e2a7aef973da0fa6c2
🧪 View experiment at: https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow/#/experiments/3


Epoch 001 | Loss: 0.2619 | Train Dice: 0.7260 | Val Dice: 0.7758 | Val IoU: 0.6545 | LR: 0.000999
         | Necrotic: 0.7533 | Edema: 0.7875 | Enhancing: 0.7844
  💾 Best model saved! Dice: 0.7758


Epoch 002 | Loss: 0.1921 | Train Dice: 0.7862 | Val Dice: 0.7787 | Val IoU: 0.6586 | LR: 0.000997
         | Necrotic: 0.7605 | Edema: 0.7907 | Enhancing: 0.7848
  💾 Best model saved! Dice: 0.7787


Epoch 003 | Loss: 0.1807 | Train Dice: 0.7982 | Val Dice: 0.7846 | Val IoU: 0.6654 | LR: 0.000994
         | Necrotic: 0.7579 | Edema: 0.7945 | Enhancing: 0.7986
  💾 Best model saved! Dice: 0.7846


Epoch 004 | Loss: 0.1719 | Train Dice: 0.8080 | Val Dice: 0.8038 | Val IoU: 0.6879 | LR: 0.000989
         | Necrotic: 0.7829 | Edema: 0.8176 | Enhancing: 0.8057
  💾 Best model saved! Dice: 0.8038


...

Epoch 091 | Loss: 0.1135 | Train Dice: 0.8709 | Val Dice: 0.8683 | Val IoU: 0.7758 | LR: 0.000527
         | Necrotic: 0.8563 | Edema: 0.8799 | Enhancing: 0.8626


Epoch 092 | Loss: 0.1133 | Train Dice: 0.8712 | Val Dice: 0.8617 | Val IoU: 0.7658 | LR: 0.000553
         | Necrotic: 0.8476 | Edema: 0.8742 | Enhancing: 0.8574


Epoch 093 | Loss: 0.1135 | Train Dice: 0.8709 | Val Dice: 0.8667 | Val IoU: 0.7733 | LR: 0.000579
         | Necrotic: 0.8552 | Edema: 0.8792 | Enhancing: 0.8596


Epoch 094 | Loss: 0.1138 | Train Dice: 0.8706 | Val Dice: 0.8613 | Val IoU: 0.7653 | LR: 0.000604
         | Necrotic: 0.8497 | Edema: 0.8747 | Enhancing: 0.8537


Epoch 95/100 Train:  91%|█████████ | 679/748 [02:56<00:17,  3.85it/s]